# 🏠 Airbnb NYC 2019 – Sentiment Analysis
## Notebook 2: Machine Learning Classifiers

**Pipeline:**
1. Load cleaned dataset
2. TF-IDF vectorization (unigrams + bigrams)
3. Train 6 classifiers: Logistic Regression, Random Forest, SVM, Gradient Boosting, KNN, AdaBoost
4. Compare metrics: Accuracy, F1, ROC-AUC
5. Confusion matrices & feature importance
6. Save best model + vectorizer

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pathlib import Path
import pickle

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, roc_auc_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier

BASE    = Path('.')
CLEANED = BASE / 'Dataset' / 'Cleaned Data'
MODELS  = BASE / 'Models'
MODELS.mkdir(exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': '#0f0f1a',
                     'axes.facecolor': '#1a1a2e', 'axes.labelcolor': 'white',
                     'xtick.color': 'white', 'ytick.color': 'white',
                     'text.color': 'white'})
PALETTE = {'Positive': '#00e5ff', 'Neutral': '#ffd54f', 'Negative': '#ff5252'}
print('✅ Imports ready')

---
### 1. Load Cleaned Data

In [ ]:
df = pd.read_csv(CLEANED / 'reviews_cleaned.csv')
df.dropna(subset=['cleaned_text', 'sentiment'], inplace=True)
print(df['sentiment'].value_counts())
df.shape

In [ ]:
X = df['cleaned_text']
y = df['sentiment']

le = LabelEncoder()
y_enc = le.fit_transform(y)   # Negative=0, Neutral=1, Positive=2

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

---
### 2. TF-IDF Vectorization

In [ ]:
tfidf = TfidfVectorizer(
    max_features=12_000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
print(f'TF-IDF matrix: {X_train_tfidf.shape}')

---
### 3. Train & Compare Models

In [ ]:
MODELS_DEF = {
    'Logistic Regression':  LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Linear SVM':           LinearSVC(max_iter=2000, C=1.0, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=100, random_state=42),
    'AdaBoost':             AdaBoostClassifier(n_estimators=100, random_state=42),
    'KNN':                  KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
}

results = []
trained_models = {}

for name, clf in MODELS_DEF.items():
    clf.fit(X_train_tfidf, y_train)
    y_pred = clf.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')
    results.append({'Model': name, 'Accuracy': acc, 'F1 (weighted)': f1})
    trained_models[name] = (clf, y_pred)
    print(f'{name:<25}  Acc={acc:.4f}  F1={f1:.4f}')

results_df = pd.DataFrame(results).sort_values('F1 (weighted)', ascending=False)
results_df

In [ ]:
# ── Performance comparison chart ─────────────────────────────────────────────
fig = px.bar(
    results_df.melt(id_vars='Model', var_name='Metric', value_name='Score'),
    x='Model', y='Score', color='Metric', barmode='group',
    title='<b>Model Comparison – Accuracy & F1</b>',
    template='plotly_dark', color_discrete_sequence=['#00e5ff', '#ffd54f']
)
fig.update_layout(title_font_size=18, yaxis_range=[0, 1])
fig.show()

---
### 4. Best Model – Detailed Evaluation

In [ ]:
best_name = results_df.iloc[0]['Model']
best_clf, best_pred = trained_models[best_name]
print(f'Best model: {best_name}')
print(classification_report(y_test, best_pred, target_names=le.classes_))

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, best_pred)
fig = px.imshow(
    cm, text_auto=True,
    x=le.classes_, y=le.classes_,
    color_continuous_scale='Blues',
    title=f'<b>Confusion Matrix – {best_name}</b>',
    template='plotly_dark'
)
fig.update_layout(xaxis_title='Predicted', yaxis_title='Actual',
                  title_font_size=16)
fig.show()

In [ ]:
# ── Top TF-IDF features per class (Logistic Regression) ──────────────────────
lr_clf = trained_models['Logistic Regression'][0]
feature_names = np.array(tfidf.get_feature_names_out())

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.patch.set_facecolor('#0f0f1a')
colors = ['#ff5252', '#ffd54f', '#00e5ff']

for i, (ax, cls) in enumerate(zip(axes, le.classes_)):
    top_idx = np.argsort(lr_clf.coef_[i])[-15:]
    ax.barh(feature_names[top_idx], lr_clf.coef_[i][top_idx],
            color=colors[i], alpha=0.85)
    ax.set_title(f'Top Features – {cls}', color=colors[i], fontsize=12)
    ax.set_xlabel('Coefficient')

plt.tight_layout()
plt.savefig('Images/feature_importance.png', bbox_inches='tight',
            facecolor='#0f0f1a', dpi=150)
plt.show()

---
### 5. Save Models

In [ ]:
# Save TF-IDF vectorizer
with open(MODELS / 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# Save all models
model_map = {
    'logistic_regression_model.pkl': trained_models['Logistic Regression'][0],
    'c_support_vector_model.pkl':     trained_models['Linear SVM'][0],
    'random_forest_model.pkl':        trained_models['Random Forest'][0],
    'gradient_boosting_model.pkl':    trained_models['Gradient Boosting'][0],
    'ada_boost_model.pkl':            trained_models['AdaBoost'][0],
    'k_neighbors_model.pkl':          trained_models['KNN'][0],
    'best_model.pkl':                  best_clf,
    'label_encoder.pkl':              le,
}
for fname, obj in model_map.items():
    with open(MODELS / fname, 'wb') as f:
        pickle.dump(obj, f)

print('✅ All models saved to Models/')